In [3]:
import os
import json
import cv2
import torch
import networkx as nx
import pandas as pd
import numpy as np

from tqdm import tqdm

import torchvision.transforms as transforms
import torchvision.models as models

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

from torch.nn import Linear

import torch.nn.functional as F

In [4]:
dataset_df = pd.read_csv(
    "../data/processed/full_composition_dataset.csv"
)

In [5]:
dataset_df["balance_target"] = np.random.rand(
    len(dataset_df)
)

dataset_df["symmetry_target"] = np.random.rand(
    len(dataset_df)
)

dataset_df["tension_target"] = np.random.rand(
    len(dataset_df)
)

dataset_df["composition_class"] = np.random.randint(
    0,
    5,
    len(dataset_df)
)

In [6]:
transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485,0.456,0.406],

        std=[0.229,0.224,0.225]
    )
])

In [7]:
cnn_backbone = models.resnet18(
    weights="DEFAULT"
)

cnn_backbone.fc = torch.nn.Identity()

In [8]:
def create_multitask_sample(

    image_path,
    graph_path,

    score_target,

    balance_target,

    symmetry_target,

    tension_target,

    class_target
):

    try:

        # LOAD IMAGE

        img = cv2.imread(image_path)

        if img is None:
            return None

        img = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        image_tensor = transform(img)

        # LOAD GRAPH

        with open(graph_path, "r") as f:

            graph_data = json.load(f)

        G = nx.node_link_graph(graph_data)

        # NODE FEATURES

        node_features = []

        valid_nodes = []

        for node, attrs in G.nodes(data=True):

            try:

                feature_vector = [

                    float(attrs.get("center_x", 0)),

                    float(attrs.get("center_y", 0)),

                    float(attrs.get("area", 0)),

                    float(attrs.get("confidence", 0))

                ]

                node_features.append(feature_vector)

                valid_nodes.append(node)

            except:
                continue

        # HANDLE EMPTY GRAPH

        if len(node_features) == 0:

            return None

        # CREATE NODE FEATURE TENSOR

        x = torch.tensor(
            node_features,
            dtype=torch.float
        )

        # CREATE EDGE INDEX

        edge_index = []

        node_mapping = {

            old_idx: new_idx
            for new_idx, old_idx in enumerate(valid_nodes)
        }

        for u, v in G.edges():

            if u in node_mapping and v in node_mapping:

                edge_index.append([
                    node_mapping[u],
                    node_mapping[v]
                ])

                edge_index.append([
                    node_mapping[v],
                    node_mapping[u]
                ])

        # HANDLE EMPTY EDGES

        if len(edge_index) == 0:

            edge_index = [[0,0]]

        edge_index = torch.tensor(
            edge_index,
            dtype=torch.long
        ).t().contiguous()

        # CREATE DATA OBJECT

        data = Data(

            x=x,

            edge_index=edge_index
        )

        # ADD IMAGE

        data.image = image_tensor

        # ADD TARGETS

        data.score_target = torch.tensor(
            [score_target],
            dtype=torch.float
        )

        data.balance_target = torch.tensor(
            [balance_target],
            dtype=torch.float
        )

        data.symmetry_target = torch.tensor(
            [symmetry_target],
            dtype=torch.float
        )

        data.tension_target = torch.tensor(
            [tension_target],
            dtype=torch.float
        )

        data.class_target = torch.tensor(
            class_target,
            dtype=torch.long
        )

        return data

    except Exception as e:

        print(
            f"Error processing {image_path}: {e}"
        )

        return None

In [9]:
IMAGE_DIR = "../data/raw/CADB/images"

In [10]:
multitask_dataset = []
subset = dataset_df
for idx, row in tqdm(subset.iterrows()):
    image_name = row["image_name"]
    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )

    graph_path = (
        "../data/processed/graphs/"
        + image_name
        + ".json"
    )
    if not os.path.exists(graph_path):

        continue
    try:

        sample = create_multitask_sample(

            image_path,

            graph_path,

            row["edge_density"],

            row["balance_target"],

            row["symmetry_target"],

            row["tension_target"],

            row["composition_class"]
        )
        if sample is not None:

            multitask_dataset.append(sample)

    except Exception as e:

        print(image_name, e)

9497it [02:46, 56.99it/s]


In [11]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(

    multitask_dataset,

    test_size=0.2,

    random_state=42
)
train_loader = DataLoader(

    train_data,

    batch_size=8,

    shuffle=True
)

test_loader = DataLoader(

    test_data,

    batch_size=8
)


In [12]:
class MultiTaskComposeNet(torch.nn.Module):
        def __init__(self):

          super().__init__()
          self.cnn = cnn_backbone
          self.conv1 = GCNConv(4,32)

          self.conv2 = GCNConv(32,64)
          self.shared_fc = Linear(
            512 + 64,
            128
          )
          self.score_head = Linear(128,1)

          self.balance_head = Linear(128,1)

          self.symmetry_head = Linear(128,1)

          self.tension_head = Linear(128,1)
          self.class_head = Linear(128,5)
        def forward(self, data):
                # IMAGE BRANCH
                batch_size = data.num_graphs
                images = data.image.view(
                  batch_size,
                  3,
                  224,
                  224
                )


                cnn_features = self.cnn(images)
                x = data.x

                edge_index = data.edge_index

                batch = data.batch
                x = self.conv1(x, edge_index)

                x = F.relu(x)

                x = self.conv2(x, edge_index)

                x = F.relu(x)
                graph_features = global_mean_pool(
                    x,
                    batch
                )
                combined = torch.cat(

                    [cnn_features, graph_features],

                    dim=1
                )
                shared = self.shared_fc(combined)

                shared = F.relu(shared)
                score_out = self.score_head(shared)

                balance_out = self.balance_head(shared)

                symmetry_out = self.symmetry_head(shared)

                tension_out = self.tension_head(shared)

                class_out = self.class_head(shared)
                return {

                    "score": score_out,

                    "balance": balance_out,

                    "symmetry": symmetry_out,

                    "tension": tension_out,

                    "class": class_out
                }


In [13]:
# STEP 14.11 — INITIALIZE MODEL

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = MultiTaskComposeNet().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

regression_loss = torch.nn.MSELoss()

classification_loss = torch.nn.CrossEntropyLoss()

print("Model Initialized Successfully")
print("Using Device:", device)

Model Initialized Successfully
Using Device: cpu


In [14]:
# STEP 14.12 — TRAINING LOOP

epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        outputs = model(batch)

        score_loss = regression_loss(
            outputs["score"].squeeze(),
            batch.score_target.squeeze()
        )

        balance_loss = regression_loss(
            outputs["balance"].squeeze(),
            batch.balance_target.squeeze()
        )

        symmetry_loss = regression_loss(
            outputs["symmetry"].squeeze(),
            batch.symmetry_target.squeeze()
        )

        tension_loss = regression_loss(
            outputs["tension"].squeeze(),
            batch.tension_target.squeeze()
        )

        class_loss = classification_loss(
            outputs["class"],
            batch.class_target
        )

        loss = (
            score_loss
            + balance_loss
            + symmetry_loss
            + tension_loss
            + class_loss
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Total Loss: {avg_loss:.4f}"
    )

Epoch 1/10 | Total Loss: 38620.2797
Epoch 2/10 | Total Loss: 1536.4886
Epoch 3/10 | Total Loss: 1399.0936
Epoch 4/10 | Total Loss: 16735.0346
Epoch 5/10 | Total Loss: 6.1530
Epoch 6/10 | Total Loss: 15.6364
Epoch 7/10 | Total Loss: 631.3249
Epoch 8/10 | Total Loss: 21.6918
Epoch 9/10 | Total Loss: 100.8724
Epoch 10/10 | Total Loss: 227.4700


In [15]:
checkpoint_path = (
    "../checkpoints/"
    "multitask_compose_net.pth"
)

torch.save({

    "model_state_dict":
        model.state_dict(),

    "optimizer_state_dict":
        optimizer.state_dict(),

    "epochs":
        epochs

}, checkpoint_path)

print("Model saved successfully")

Model saved successfully
